# Análisis de Red de Tiendas RetailNow

Uso de Pandas y Numpy para analizar ventas, inventarios y satisfacción del cliente

In [22]:
import pandas as pd
import numpy as np

sales = pd.read_csv('/workspace/sales.csv')
inventories = pd.read_csv('/workspace/inventories.csv')
satisfaction = pd.read_csv('/workspace/satisfaction.csv')

sales = sales.dropna()
inventories = inventories.dropna()
satisfaction = satisfaction.dropna()

print(sales.shape, inventories.shape, satisfaction.shape)
sales.head()

(10, 5) (10, 4) (5, 3)


,ID_Tienda,Producto,Cantidad_Vendida,Precio_Unitario,Fecha_Venta
0,1,Producto A,20,100,2023-01-05
1,1,Producto B,15,200,2023-01-06
2,2,Producto A,30,100,2023-01-07
3,2,Producto C,25,300,2023-01-08
4,3,Producto A,10,100,2023-01-09


## Análisis de Ventas

In [11]:
sales['Total_Ventas'] = sales['Cantidad_Vendida'] * sales['Precio_Unitario']

ventas_producto = sales.groupby('Producto')['Cantidad_Vendida'].sum()
ventas_tienda = sales.groupby('ID_Tienda')['Cantidad_Vendida'].sum()
ingresos_tienda = sales.groupby('ID_Tienda')['Total_Ventas'].sum()

print(ventas_producto)
print(ventas_tienda)
print(ingresos_tienda)

sales.describe()

Producto
Producto A    85
Producto B    75
Producto C    90
Name: Cantidad_Vendida, dtype: int64
ID_Tienda
1    35
2    55
3    50
4    60
5    50
Name: Cantidad_Vendida, dtype: int64
ID_Tienda
1     5000
2    10500
3     9000
4    13000
5    13000
Name: Total_Ventas, dtype: int64


,ID_Tienda,Cantidad_Vendida,Precio_Unitario,Total_Ventas
count,10.000000,10.000000,10.000000,10.000000
mean,3.000000,25.000000,190.000000,5050.000000
std,1.490712,9.128709,87.559504,3361.960407
min,1.000000,10.000000,100.000000,1000.000000
25%,2.000000,20.000000,100.000000,2625.000000
50%,3.000000,25.000000,200.000000,3500.000000
75%,4.000000,30.000000,275.000000,7875.000000
max,5.000000,40.000000,300.000000,10500.000000


## Rotación de Inventarios

In [12]:
ventas_tp = sales.groupby(['ID_Tienda','Producto'])['Cantidad_Vendida'].sum().reset_index()

inventario_rotacion = inventories.merge(
    ventas_tp,
    on=['ID_Tienda','Producto'],
    how='left'
)

inventario_rotacion['Cantidad_Vendida'] = inventario_rotacion['Cantidad_Vendida'].fillna(0)

inventario_rotacion['Rotacion_Inventario'] = (
    inventario_rotacion['Cantidad_Vendida'] / inventario_rotacion['Stock_Disponible']
)

inventario_rotacion.head()

,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización,Cantidad_Vendida,Rotacion_Inventario
0,1,Producto A,50,2023-01-05,20,0.400000
1,1,Producto B,40,2023-01-06,15,0.375000
2,2,Producto A,60,2023-01-07,30,0.500000
3,2,Producto C,45,2023-01-08,25,0.555556
4,3,Producto A,30,2023-01-09,10,0.333333


In [13]:
criticos = inventario_rotacion[
    inventario_rotacion['Rotacion_Inventario'] < 0.10
]

criticos

,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización,Cantidad_Vendida,Rotacion_Inventario


## Satisfacción del Cliente

In [14]:
baja_satisfaccion = satisfaction[
    satisfaction['Satisfacción_Promedio'] < 60
]

baja_satisfaccion

,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación
4,5,55,2023-01-15


In [15]:
ventas_totales_tienda = sales.groupby('ID_Tienda')['Total_Ventas'].sum().reset_index()

analisis_satisfaccion = ventas_totales_tienda.merge(
    satisfaction,
    on = 'ID_Tienda'
)

analisis_satisfaccion

,ID_Tienda,Total_Ventas,Satisfacción_Promedio,Fecha_Evaluación
0,1,5000,85,2023-01-15
1,2,10500,90,2023-01-15
2,3,9000,70,2023-01-15
3,4,13000,65,2023-01-15
4,5,13000,55,2023-01-15


## Operaciones con Numpy

In [16]:
ventas_np = sales['Total_Ventas'].to_numpy()

mediana = np.median(ventas_np)
desviacion = np.std(ventas_np)

print('Mediana:', mediana)
print('Desviación estándar:', desviacion)

Mediana: 3500.0
Desviación estándar: 3189.4356867634124


In [17]:
np.random.seed(42)

promedio = np.mean(ventas_np)
desv = np.std(ventas_np)

proyecciones = np.random.normal(promedio, desv, 12)

print(proyecciones)
print('Promedio:', np.mean(proyecciones))
print('Máximo:', np.max(proyecciones))
print('Mínimo:', np.min(proyecciones))

[ 6634.23784573  4609.01490364  7115.76093733  9907.60577603
  4303.18287048  4303.23523392 10086.79771077  7497.68371242
  3552.64163948  6780.46036522  3571.95907267  3564.58490358]
Promedio: 5993.930414272573
Máximo: 10086.7977107734
Mínimo: 3552.6416394777248
